# Rastreabilidade Assistencial no SUS via Data Linkage
### Auditoria da População de Pacientes (jul de 2024 à jun de 2025) entre RNDS e SIA/SIH utilizando CPF/CNS como Identificador Único”

Avaliação da integridade e completude do fluxo de dados assistenciais no SUS, verificando a sobreposição e as exclusões (pacientes) entre os sistemas de Regulação (RNDS) e Faturamento (SIA/SIH) para o ano de 2024. A análise utiliza o CPF ou CNS como chave de linkage, alinhando-se aos princípios da Portaria 6.656/2025.

### Objetivo Principal:
Demonstrar a porcentagem de sobreposição de pacientes e procedimentos entre as bases. Especificamente:
•	Comprovar a rastreabilidade: Determinar a proporção de pacientes concluídos na RNDS (Regulação) que efetivamente aparecem nas bases de faturamento (SIA/SIH).
•	Identificar gargalos/inconsistências: Determinar a proporção de pacientes faturados (SIA/SIH) que não passaram ou não tiveram registro de conclusão na RNDS, evidenciando falhas no registro de Regulação Assistencial.
•	Validar a RNDS: Comprovar estatisticamente que os dados da RNDS, apesar de iniciais, já fornecem uma base válida para estudos de fluxo assistencial e tempo de espera.


### Metodologia de Ciência de Dados e Estatística (Foco em Linkage):
#### ETAPA 1: Analise e tratamento.
Devido ao volume de dados os arquivos estão em Parquet, veja o 'convert_csv_parquet.ipynb' 


In [ ]:
# BIBLIOTECA
import pyarrow.parquet as pq
import pyarrow.compute as pc

In [ ]:
# Resumo das informações dos bancos 
def resumo_identificacao_parquet(parquet_path, batch_size=500_000):
    parquet_file = pq.ParquetFile(parquet_path)

    total = sem_cpf = sem_cns = sem_ambos = 0

    for batch in parquet_file.iter_batches(
        batch_size=batch_size,
        columns=["CPF_PAC", "CNS_PAC"]
    ):
        cpf_vazio = pc.equal(batch["CPF_PAC"], "")
        cns_vazio = pc.equal(batch["CNS_PAC"], "")

        total += batch.num_rows
        sem_cpf += pc.sum(cpf_vazio).as_py()
        sem_cns += pc.sum(cns_vazio).as_py()
        sem_ambos += pc.sum(pc.and_(cpf_vazio, cns_vazio)).as_py()

    return {
        "GERAL": total,
        "SEM CPF": sem_cpf,
        "SEM CNS": sem_cns,
        "SEM CPF e SEM CNS": sem_ambos
    }


In [ ]:
print("===== BASE SIH =====")
resumo_sih = resumo_identificacao_parquet(r"base\SIH.parquet")

for k, v in resumo_sih.items():
    print(f"{k:<25} {v:,}")


In [ ]:
print("===== BASE SIA =====")
resumo_sia = resumo_identificacao_parquet(r"base\SIA.parquet")

for k, v in resumo_sia.items():
    print(f"{k:<25} {v:,}")


## BASE DA REGULAÇÃO
neste caso temos que primeiro trocar os nomes das colunas para padronizar com as bases SIH e SIA

In [ ]:
import pyarrow.parquet as pq
import pyarrow as pa

# Caminhos
arquivo_in = r"base\RNDS.parquet"
arquivo_out = r"base\RNDS_renomeado.parquet"

# Dicionário de renomeação
mapa_colunas = {
    "nu_cpf_paciente": "CPF_PAC",
    "nu_cns_paciente": "CNS_PAC",
    "co_sigtap": "COD_SIGTAP_PROCEDIMENTO",
    "co_cbo": "CBO",
    "sg_uf_estab_executante": "UF_DESC_ATEND",
    "co_municipio_estab_executante": "IBGE_ATEND",
    "co_cnes_estab_executante": "CNES_ATEND",
    "data_solicitacao": "DATA_SOLICITACAO",
    "data_autorizacao": "DATA_AUTORIZACAO",
    "data_execucao": "DATA_EXECUCAO",
    "st_vida_paciente": "ST_VIDA",
    "st_solicitacao": "STATUS",
    "id_registro_sistema_origem": "ID_ORIGEM",
    "ds_sistema_origem": "SIST_ORIGEM"
}

# Abre o parquet original
parquet_file = pq.ParquetFile(arquivo_in)

# Novo writer
writer = None

# Processa em lotes (não carrega tudo na memória)
for batch in parquet_file.iter_batches(batch_size=500_000):
    table = pa.Table.from_batches([batch])
    # Renomeia colunas
    table = table.rename_columns([mapa_colunas.get(c, c) for c in table.schema.names])
    if writer is None:
        writer = pq.ParquetWriter(arquivo_out, table.schema)
    writer.write_table(table)

if writer:
    writer.close()

print("✅ Arquivo renomeado salvo em:", arquivo_out)


In [ ]:
print("===== BASE RNDS =====")
resumo_rnds = resumo_identificacao_parquet(r"base\RNDS_renomeado.parquet")

for k, v in resumo_rnds.items():
    print(f"{k:<25} {v:,}")


In [ ]:
# Bibliotecas 
import pyarrow.parquet as pq
import pyarrow.compute as pc


# resumo de saida 
def resumo_identificacao(table):
    return {
        "GERAL": table.num_rows,
        "SEM CPF": pc.filter(table, pc.equal(table["CPF_PAC"], "")).num_rows,
        "SEM CNS": pc.filter(table, pc.equal(table["CNS_PAC"], "")).num_rows,
        "SEM CPF e SEM CNS": pc.filter(
            table,
            pc.and_(
                pc.equal(table["CPF_PAC"], ""),
                pc.equal(table["CNS_PAC"], "")
            )
        ).num_rows
    }


In [ ]:
# Carregamendo dos dados (banco de dados)
table_sia = pq.read_table(r"base\SIA.parquet")
table_sih = pq.read_table(r"base\SIH.parquet")



In [ ]:
# BASE HOSPITALAR
print("===== BASE SIH =====")
resumo_sih = resumo_identificacao(table_sih)

for k, v in resumo_sih.items():
    print(f"{k:<20} {v:,}")


In [ ]:
# BASE AMBULATORIAL
print("\n===== BASE SIA =====")
resumo_sia = resumo_identificacao(table_sia)

for k, v in resumo_sia.items():
    print(f"{k:<20} {v:,}")


In [ ]:
# RETIRAR TODAS AS SOLICITAÇÕES 
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

def limpar_parquet_sem_cpf_e_cns(
    parquet_in: str,
    parquet_out: str,
    schema: pa.Schema,
    compression: str = "zstd",
):
    """
    Remove linhas onde CPF_PAC = '' E CNS_PAC = ''
    """
    parquet_file = pq.ParquetFile(parquet_in)
    writer = None

    try:
        for batch in parquet_file.iter_batches():
            table = pa.Table.from_batches([batch])

            # condição: CPF vazio E CNS vazio
            cond_remover = pc.and_(
                pc.equal(table["CPF_PAC"], ""),
                pc.equal(table["CNS_PAC"], "")
            )

            # manter o inverso
            table_limpa = pc.filter(table, pc.invert(cond_remover))

            if table_limpa.num_rows == 0:
                continue

            if writer is None:
                writer = pq.ParquetWriter(
                    parquet_out,
                    schema=schema,
                    compression=compression,
                    use_dictionary=True,
                )

            writer.write_table(table_limpa)

    finally:
        if writer is not None:
            writer.close()


In [ ]:
# PRIMEIRA LIMPESA DE BASE HOSPITALAR
schema_sih = pq.read_schema(r"base\SIH.parquet")

limpar_parquet_sem_cpf_e_cns(
    parquet_in=r"base\SIH.parquet",
    parquet_out=r"base\SIH_LIMPO.parquet",
    schema=schema_sih,
)

print("✅ SIH limpo criado")


In [ ]:
# PRIMEIRA LIMPESA DE BASE AMBULATORIAL
schema_sia = pq.read_schema(r"base\SIA.parquet")

limpar_parquet_sem_cpf_e_cns(
    parquet_in=r"base\SIA.parquet",
    parquet_out=r"base\SIA_LIMPO.parquet",
    schema=schema_sia,
)

print("✅ SIA limpo criado")


In [ ]:
# PRIMEIRA LIMPESA DE BASE REGULAÇAO 
schema_sih = pq.read_schema(r"base\RNDS.parquet")

limpar_parquet_sem_cpf_e_cns(
    parquet_in=r"base\RNDS.parquet",
    parquet_out=r"base\RNDS_LIMPO.parquet",
    schema=schema_sih,
)

print("✅ RNDS limpo criado")


## CRIAÇÃO DE CHAVE UNICA --- USO DO CPF

Agora a ideia é primeiro localizar o CPF das solicitações HOSPITLAR E AMBULATORIAL. 

In [ ]:
# -----------------------------------------------------
# -------------CRIAÇÃO DE CHAVES UNICA ----------------



In [ ]:
table_rnds = pq.read_table(r"base\RNDS.parquet")

In [ ]:
# BASE REGULAÇÃO 
print("\n===== BASE RNDS =====")
resumo_rnds = resumo_identificacao(table_rnds)

for k, v in resumo_rnds.items():
    print(f"{k:<20} {v:,}")